# ==========================================================
# Breast Cancer Survival Prediction using Apache Spark
# Notebook 04: Model Training & Evaluation
# ==========================================================

"""
Objective
---------
1. Load the processed feature dataset from Notebook 03.
2. Split dataset into Training (80%) and Testing (20%) sets.
3. Compute dynamic class weights to address data imbalance.
4. Construct dual Apache Spark ML Pipelines (Tree vs Linear).
5. Train four different models:
   - Logistic Regression
   - Decision Tree
   - Random Forest
   - Gradient-Boosted Trees (GBT)
6. Perform baseline validation (ROC-AUC) before exporting trained models & test sets.
"""

1. THIẾT LẬP VÀ KHẢO SÁT BAN ĐẦU

In [1]:
# 1. Import Libraries | Khai báo thư viện

print("=" * 60)
print("1. IMPORT LIBRARIES")
print("=" * 60)

import os
import sys
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Add project root directory to path | Thêm thư mục gốc dự án vào hệ thống
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# Import custom functions from src | Nạp các hàm tự định nghĩa từ src
from src.data.loader import create_spark_session, load_csv

1. IMPORT LIBRARIES


In [2]:
# 2. Create Spark Session | Khởi tạo phiên làm việc Spark

print("=" * 60)
print("2. CREATE SPARK SESSION")
print("=" * 60)

spark = create_spark_session("SEER Breast Cancer Model Training")

2. CREATE SPARK SESSION


In [3]:
# 3: Load Feature Dataset | Tải dữ liệu thuộc tính từ Notebook 03

print("\n" + "=" * 60)
print("3. LOAD FEATURE DATASET (CLEAN)")
print("=" * 60)

# Load trực tiếp vì Notebook 03 đã đảm bảo định dạng đúng
df = load_csv(spark, "../data/processed/seer_breast_cancer_feature.csv")


3. LOAD FEATURE DATASET (CLEAN)


In [4]:
# 4: Dataset Overview & Schema | Khảo sát cấu trúc và lược đồ dữ liệu

print("\n" + "=" * 60)
print("4. DATASET OVERVIEW & SCHEMA")
print("=" * 60)

print(f"Total Rows    : {df.count():,}")
print(f"Total Columns : {len(df.columns)}")
df.printSchema()


4. DATASET OVERVIEW & SCHEMA
Total Rows    : 456,087
Total Columns : 26
root
 |-- Age: double (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Race: string (nullable = true)
 |-- Marital_Status: string (nullable = true)
 |-- Tumor_Size: double (nullable = true)
 |-- Grade: string (nullable = true)
 |-- AJCC_T: string (nullable = true)
 |-- AJCC_N: string (nullable = true)
 |-- Regional_Nodes_Examined: double (nullable = true)
 |-- Regional_Nodes_Positive: double (nullable = true)
 |-- Sequence_Number: string (nullable = true)
 |-- Histologic_Type: integer (nullable = true)
 |-- Laterality: string (nullable = true)
 |-- Diagnostic_Confirmation: string (nullable = true)
 |-- AJCC_M: string (nullable = true)
 |-- Surgery_Primary_Site: integer (nullable = true)
 |-- Surgery_Other_Regional: string (nullable = true)
 |-- Surgery_Radiation_Sequence: string (nullable = true)
 |-- Radiation: string (nullable = true)
 |-- Chemotherapy: string (nullable = true)
 |-- AJCC_Stage: string (

In [5]:
# 5: Preview Dataset | Hiển thị mẫu dữ liệu thực tế

print("\n" + "=" * 60)
print("5. PREVIEW DATASET")
print("=" * 60)
df.show(5, truncate=False)


5. PREVIEW DATASET
+----+------+-----+------------------------------+----------+-----------------------------------+------+------+-----------------------+-----------------------+----------------+---------------+-------------------------+-----------------------+------+--------------------+--------------------------+-------------------------------------------------------------------------+---------------+------------+----------+-----+------------+----------------+-------------------+--------------+
|Age |Sex   |Race |Marital_Status                |Tumor_Size|Grade                              |AJCC_T|AJCC_N|Regional_Nodes_Examined|Regional_Nodes_Positive|Sequence_Number |Histologic_Type|Laterality               |Diagnostic_Confirmation|AJCC_M|Surgery_Primary_Site|Surgery_Other_Regional    |Surgery_Radiation_Sequence                                               |Radiation      |Chemotherapy|AJCC_Stage|label|Age_Group   |Tumor_Size_Group|Node_Ratio         |Hormone_Status|
+----+------+-

2. PHÂN CHIA DỮ LIỆU & XỬ LÝ MẤT CÂN BẰNG (CHẶN DATA LEAKAGE)

In [6]:
# 6: Train/Test Split | Thực hiện phân chia dữ liệu huấn luyện/kiểm thử

print("\n" + "=" * 60)
print("6. TRAIN/TEST SPLIT (PREVENT DATA LEAKAGE)")
print("=" * 60)

# Secure a reproducible split using random seed | Cố định phân vùng bằng seed trước khi xử lý pipeline để chặn rò rỉ thông tin
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print(f"Training Dataset Rows : {train_df.count():,}")
print(f"Testing Dataset Rows  : {test_df.count():,}")


6. TRAIN/TEST SPLIT (PREVENT DATA LEAKAGE)
Training Dataset Rows : 365,319
Testing Dataset Rows  : 90,768


In [7]:
# 7: Check Class Distribution | Thống kê tỷ lệ nhãn Alive/Dead trên tập Train

print("\n" + "=" * 60)
print("7. CHECK CLASS DISTRIBUTION")
print("=" * 60)

class_counts = train_df.groupBy("label").count().collect()
counts = {row["label"]: row["count"] for row in class_counts}
print(f"Class distributions in Train Set: {counts}")


7. CHECK CLASS DISTRIBUTION
Class distributions in Train Set: {1: 135246, 0: 230073}


In [8]:
# 8: Compute Class Weights | Tính toán trọng số nghịch đảo và gán cột 'weight' cho tập Train

print("\n" + "=" * 60)
print("8. COMPUTE CLASS WEIGHTS")
print("=" * 60)

total_count = train_df.count()
weight_0 = total_count / (2.0 * counts[0.0])
weight_1 = total_count / (2.0 * counts[1.0])

print(f"Calculated class weights: Class 0 (Alive) = {weight_0:.4f} | Class 1 (Dead) = {weight_1:.4f}")

# Map weights into a new train column | Đổ cột trọng số thực tế vào tập Train
train_df = train_df.withColumn(
    "weight",
    F.when(F.col("label") == 1.0, weight_1).otherwise(weight_0)
)

# Test set has no weights to prevent leakage | Không gán trọng số cho tập Test nhằm chặn rò rỉ dữ liệu
test_df = test_df.withColumn("weight", F.lit(1.0))


8. COMPUTE CLASS WEIGHTS
Calculated class weights: Class 0 (Alive) = 0.7939 | Class 1 (Dead) = 1.3506


3. THIẾT LẬP PIPELINE SONG SONG (TỐI ƯU HÓA CHO SPARK CLUSTER)

In [9]:
# Define categorical and numerical features | Khai báo nhóm thuộc tính đặc trưng
# [CORRECTED] Sequence_Number removed as it was identified as a constant feature in Notebook 02
categorical_cols = [
    "Sex", "Race", "Marital_Status", "Grade", "AJCC_T", "AJCC_N", "AJCC_M", 
    "AJCC_Stage", "Laterality", "Diagnostic_Confirmation", 
    "Surgery_Other_Regional", "Surgery_Radiation_Sequence", "Radiation", 
    "Chemotherapy", "Age_Group", "Tumor_Size_Group", "Hormone_Status",
    "Histologic_Type", "Surgery_Primary_Site"
]

numerical_cols = [
    "Age", "Tumor_Size", "Regional_Nodes_Examined", "Regional_Nodes_Positive", "Node_Ratio"
]

In [10]:
# 9: StringIndexer Setup | Mã hóa nhãn phân loại thành số nguyên

print("\n" + "=" * 60)
print("9. STRINGINDEXER SETUP")
print("=" * 60)

indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_indexed", handleInvalid="keep")
    for col in categorical_cols
]
indexed_categorical_cols = [f"{col}_indexed" for col in categorical_cols]
print(f"Created StringIndexers for {len(categorical_cols)} categorical features.")


9. STRINGINDEXER SETUP
Created StringIndexers for 19 categorical features.


In [11]:
# 10: OneHotEncoder Setup | Mã hóa One-Hot từ cột Index (chỉ cho mô hình tuyến tính)

print("\n" + "=" * 60)
print("10. ONEHOTENCODER SETUP")
print("=" * 60)

encoder = OneHotEncoder(
    inputCols=indexed_categorical_cols,
    outputCols=[f"{col}_encoded" for col in categorical_cols]
)
encoded_categorical_cols = [f"{col}_encoded" for col in categorical_cols]
print("OneHotEncoder configured for linear models.")


10. ONEHOTENCODER SETUP
OneHotEncoder configured for linear models.


In [12]:
# 11: VectorAssembler Setup | Thiết lập 2 bộ Assembler độc lập

print("\n" + "=" * 60)
print("11. VECTORASSEMBLER SETUP")
print("=" * 60)

# Assembler 1: Linear Models (One-Hot Categories + Scaled Numerics)
assembler_linear = VectorAssembler(
    inputCols=encoded_categorical_cols + numerical_cols,
    outputCol="unscaled_features",
    handleInvalid="skip"
)
scaler = StandardScaler(
    inputCol="unscaled_features",
    outputCol="features",
    withStd=True,
    withMean=False
)

# Assembler 2: Tree Models (Raw Categorical Indexes + Raw Numerics to avoid dummy variables OOM)
assembler_tree = VectorAssembler(
    inputCols=indexed_categorical_cols + numerical_cols,
    outputCol="features"
)

print("Dual VectorAssembler successfully initialized.")


11. VECTORASSEMBLER SETUP
Dual VectorAssembler successfully initialized.


In [13]:
# 12: Transform Train & Test Sets | Thực thi Pipeline (Stable Version)

print("\n" + "=" * 60)
print("12. TRANSFORM TRAIN & TEST SETS (STABLE - NO CACHE)")
print("=" * 60)

# Build transformation pipelines | Biên dịch luồng Pipeline
pipeline_linear = Pipeline(stages=indexers + [encoder, assembler_linear, scaler])
pipeline_tree = Pipeline(stages=indexers + [assembler_tree])

print("Fitting transformation pipelines on Train set...")
preproc_model_linear = pipeline_linear.fit(train_df)
preproc_model_tree = pipeline_tree.fit(train_df)

# Generate final transformed datasets | Thực thi chuyển đổi cấu trúc
# Spark sẽ thực hiện các biến đổi này mỗi khi dữ liệu được gọi tới (Lazy Evaluation)

train_linear = preproc_model_linear.transform(train_df)
test_linear = preproc_model_linear.transform(test_df)

train_tree = preproc_model_tree.transform(train_df)
test_tree = preproc_model_tree.transform(test_df)

print("Transformation completed successfully.")
print(f"Ready to proceed with modeling using train_linear and train_tree.")


12. TRANSFORM TRAIN & TEST SETS (STABLE - NO CACHE)
Fitting transformation pipelines on Train set...
Transformation completed successfully.
Ready to proceed with modeling using train_linear and train_tree.


4. HUẤN LUYỆN MÔ HÌNH VÀ THỰC NGHIỆM ĐO THỜI GIAN

In [15]:
training_times = {}

In [16]:
# 13: Train Logistic Regression | Huấn luyện mô hình Logistic Regression

print("\n" + "=" * 60)
print("13. TRAIN LOGISTIC REGRESSION")
print("=" * 60)

lr = LogisticRegression(featuresCol="features", labelCol="label", weightCol="weight", maxIter=100)
start_time = time.time()
lr_model = lr.fit(train_linear)
training_times["Logistic Regression"] = time.time() - start_time
print(f"Logistic Regression trained in {training_times['Logistic Regression']:.2f} seconds.")


13. TRAIN LOGISTIC REGRESSION
Logistic Regression trained in 46.78 seconds.


In [18]:
# 14: Train Decision Tree | Huấn luyện mô hình Decision Tree

print("\n" + "=" * 60)
print("14. TRAIN DECISION TREE")
print("=" * 60)

dt = DecisionTreeClassifier(featuresCol="features", labelCol="label", weightCol="weight", seed=42, maxBins=150)
start_time = time.time()
dt_model = dt.fit(train_tree)
training_times["Decision Tree"] = time.time() - start_time
print(f"Decision Tree trained in {training_times['Decision Tree']:.2f} seconds.")


14. TRAIN DECISION TREE
Decision Tree trained in 10.88 seconds.


In [20]:
# 15: Train Random Forest | Huấn luyện mô hình Random Forest

print("\n" + "=" * 60)
print("15. TRAIN RANDOM FOREST")
print("=" * 60)

rf = RandomForestClassifier(featuresCol="features", labelCol="label", weightCol="weight", seed=42, numTrees=100, maxBins=150)
start_time = time.time()
rf_model = rf.fit(train_tree)
training_times["Random Forest"] = time.time() - start_time
print(f"Random Forest trained in {training_times['Random Forest']:.2f} seconds.")


15. TRAIN RANDOM FOREST
Random Forest trained in 31.76 seconds.


In [21]:
# 16: Train GBT Classifier | Huấn luyện mô hình GBT Classifier

print("\n" + "=" * 60)
print("16. TRAIN GBT CLASSIFIER")
print("=" * 60)

gbt = GBTClassifier(featuresCol="features", labelCol="label", seed=42, maxIter=50, maxBins=150)
start_time = time.time()
gbt_model = gbt.fit(train_tree)
training_times["GBT Classifier"] = time.time() - start_time
print(f"GBT Classifier trained in {training_times['GBT Classifier']:.2f} seconds.")


16. TRAIN GBT CLASSIFIER
GBT Classifier trained in 167.94 seconds.


In [22]:
# 17: Compare Training Time | Xuất bảng so sánh thời gian thực thi của các mô hình trên Spark

print("\n" + "=" * 60)
print("17. COMPARE TRAINING TIME")
print("=" * 60)

print(f"{'Algorithm':<25} | {'Training Time (Seconds)':<25}")
print("-" * 55)
for name, elapsed in training_times.items():
    print(f"{name:<25} | {elapsed:<25.2f}")
print("-" * 55)


17. COMPARE TRAINING TIME
Algorithm                 | Training Time (Seconds)  
-------------------------------------------------------
Logistic Regression       | 46.78                    
Decision Tree             | 10.88                    
Random Forest             | 31.76                    
GBT Classifier            | 167.94                   
-------------------------------------------------------


5. DỰ ĐOÁN, ĐÓNG GÓI VÀ BÁO CÁO HỆ THỐNG

In [23]:
# 18: Generate Logistic Regression Predictions | Dự đoán trên tập Test

print("\n" + "=" * 60)
print("18. GENERATE LOGISTIC REGRESSION PREDICTIONS")
print("=" * 60)
lr_predictions = lr_model.transform(test_linear)
print("Logistic Regression predictions generated.")


18. GENERATE LOGISTIC REGRESSION PREDICTIONS
Logistic Regression predictions generated.


In [24]:
#  19: Generate Decision Tree Predictions | Dự đoán trên tập Test

print("\n" + "=" * 60)
print("19. GENERATE DECISION TREE PREDICTIONS")
print("=" * 60)
dt_predictions = dt_model.transform(test_tree)
print("Decision Tree predictions generated.")


19. GENERATE DECISION TREE PREDICTIONS
Decision Tree predictions generated.


In [25]:
# 20: Generate Random Forest Predictions | Dự đoán trên tập Test
print("\n" + "=" * 60)
print("20. GENERATE RANDOM FOREST PREDICTIONS")
print("=" * 60)
rf_predictions = rf_model.transform(test_tree)
print("Random Forest predictions generated.")


20. GENERATE RANDOM FOREST PREDICTIONS
Random Forest predictions generated.


In [26]:
# 21: Generate GBT Predictions | Dự đoán trên tập Test

print("\n" + "=" * 60)
print("21. GENERATE GBT PREDICTIONS")
print("=" * 60)
gbt_predictions = gbt_model.transform(test_tree)
print("GBT Classifier predictions generated.")


21. GENERATE GBT PREDICTIONS
GBT Classifier predictions generated.


In [27]:
# 22: Validate Prediction Results | Kiểm tra tính toàn vẹn và khớp số lượng dòng dự đoán

print("\n" + "=" * 60)
print("22. VALIDATE PREDICTION RESULTS")
print("=" * 60)

test_row_count = test_df.count()
lr_count = lr_predictions.count()
dt_count = dt_predictions.count()
rf_count = rf_predictions.count()
gbt_count = gbt_predictions.count()

print(f"Target Testing Row Count : {test_row_count:,}")
print(f"Logistic Reg Predictions : {lr_count:,} | Status: {'MATCH' if lr_count == test_row_count else 'MISMATCH'}")
print(f"Decision Tree Predictions: {dt_count:,} | Status: {'MATCH' if dt_count == test_row_count else 'MISMATCH'}")
print(f"Random Forest Predictions: {rf_count:,} | Status: {'MATCH' if rf_count == test_row_count else 'MISMATCH'}")
print(f"GBT Classifier Preds     : {gbt_count:,} | Status: {'MATCH' if gbt_count == test_row_count else 'MISMATCH'}")


22. VALIDATE PREDICTION RESULTS
Target Testing Row Count : 90,768
Logistic Reg Predictions : 90,768 | Status: MATCH
Decision Tree Predictions: 90,768 | Status: MATCH
Random Forest Predictions: 90,768 | Status: MATCH
GBT Classifier Preds     : 90,768 | Status: MATCH


In [ ]:
# 23: Save Trained Models, Pipelines & Frozen Test Dataset
import os

print("\n" + "=" * 60)
print("23. SAVE TRAINED MODELS & PIPELINES")
print("=" * 60)

os.makedirs("../models", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

# Danh sách các model để lưu thử
models = {
    "logistic_regression": lr_model,
    "decision_tree": dt_model,
    "random_forest": rf_model,
    "gbt_classifier": gbt_model
}

# Lưu Model
for name, model in models.items():
    try:
        model.save(f"../models/{name}_model")
        print(f"✓ Saved {name}_model")
    except Exception as e:
        print(f"x Skipping {name}_model (Lỗi hệ thống tệp trên Windows: {e})")

# Lưu Pipelines
try:
    preproc_model_linear.save("../models/logistic_regression_pipeline")
    preproc_model_tree.save("../models/tree_models_pipeline")
    print("✓ Pipelines saved successfully")
except Exception as e:
    print(f"x Skipping pipelines (Lỗi hệ thống tệp trên Windows: {e})")

# Lưu Test Dataset (Dùng Pandas thay thế Parquet để tránh lỗi Hadoop)
try:
    test_df.toPandas().to_parquet("../data/processed/test_dataset.parquet")
    print("✓ Frozen test dataset saved as Parquet via Pandas")
except Exception as e:
    print(f"x Skipping test_dataset saving: {e}")

print("\n--- Hoàn tất quy trình ---")


23. SAVE TRAINED MODELS & PIPELINES
x Skipping logistic_regression_model (Lỗi hệ thống tệp trên Windows: An error occurred while calling o4908.save.
: java.io.IOException: Path ../models/logistic_regression_model already exists. To overwrite it, please use write.overwrite().save(path) for Scala and use write().overwrite().save(path) for Java and Python.
	at org.apache.spark.ml.util.FileSystemOverwrite.handleOverwrite(ReadWrite.scala:683)
	at org.apache.spark.ml.util.MLWriter.save(ReadWrite.scala:167)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:77)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:569)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
	

In [31]:
# 24: Model Training Report & Baseline Validation

print("\n" + "=" * 60)
print("24. MODEL TRAINING REPORT & BASELINE VALIDATION")
print("=" * 60)

evaluator_auc = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

print("""
================================================================================
FINAL MODEL TRAINING PIPELINE COMPLETED
================================================================================
✓ Train/Test Split correctly isolated.
✓ Balanced class representation handled dynamically.
✓ Models trained successfully without memory-intensive caching.
""")

print(f"Logistic Regression Baseline ROC-AUC: {evaluator_auc.evaluate(lr_predictions):.4f}")
print(f"Decision Tree Baseline ROC-AUC      : {evaluator_auc.evaluate(dt_predictions):.4f}")
print(f"Random Forest Baseline ROC-AUC      : {evaluator_auc.evaluate(rf_predictions):.4f}")
print(f"GBT Classifier Baseline ROC-AUC     : {evaluator_auc.evaluate(gbt_predictions):.4f}")

# ĐÃ BỎ HẾT LỆNH UNPERSIST ĐỂ TRÁNH LỖI KHI KHÔNG DÙNG CACHE
print("\nReady for Notebook 05 — Advanced Model Evaluation!")


24. MODEL TRAINING REPORT & BASELINE VALIDATION

FINAL MODEL TRAINING PIPELINE COMPLETED
✓ Train/Test Split correctly isolated.
✓ Balanced class representation handled dynamically.
✓ Models trained successfully without memory-intensive caching.

Logistic Regression Baseline ROC-AUC: 0.8438
Decision Tree Baseline ROC-AUC      : 0.4819
Random Forest Baseline ROC-AUC      : 0.8316
GBT Classifier Baseline ROC-AUC     : 0.8569

Ready for Notebook 05 — Advanced Model Evaluation!
